In [ ]:
import pandas as pd
from pathlib import Path

# Base directory
base_dir = Path(r"C..\res_rq3\out_s_ela")

# Dimensions (replace {d} with actual value if needed)
d = 2  

# Generate all prefixes smartly: 111, 112, 121, 122, 211, 212, 221, 222
prefixes = [f"{i}{j}{k}" for i in [1,2] for j in [1,2] for k in [1,2]]

# List to store DataFrames
df_list = []

for p in prefixes:
    file_path = base_dir / f"s_ela_msg_{p}_{d}d_median.csv"
    if file_path.exists():
        df = pd.read_csv(file_path)
        df_list.append(df)
    else:
        print(f"Warning: {file_path} not found, skipping.")

# Concatenate all DataFrames vertically
merged_sela_df = pd.concat(df_list, axis=0, ignore_index=True)

# Optional: Save merged DataFrame
merged_sela_df.to_csv(base_dir / f"s_ela_msg_all_{d}d_median.csv", index=False)

print("Merged S-ELA files shape:", merged_sela_df.shape)


In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib import rcParams
import numpy as np

# ----------------------------------------
# Font settings for plotting
# ----------------------------------------
rcParams["text.usetex"] = False
rcParams["mathtext.fontset"] = "cm"
rcParams["pdf.fonttype"] = 42
rcParams["ps.fonttype"] = 42
rcParams["figure.dpi"] = 300
rcParams["savefig.dpi"] = 300

# ----------------------------------------
# 1. Load data
# ----------------------------------------
ela_df = pd.read_csv(r'..\res_rq2\msg_ela\ela_msg_2d_median.csv')
sela_df = pd.read_csv(r'..\res_rq3\out_s_ela\s_ela_msg_3d_median.csv')

# ----------------------------------------
# 2. Load stable S-ELA features
# ----------------------------------------
stable_sela_features = pd.read_csv(r'..\res_rq2\stable_s_ela_features_2d.csv')['feature'].tolist()

# ----------------------------------------
# 3. Merge datasets on common keys (inner join)
# Keeps only rows existing in both datasets
# ----------------------------------------
merged_df = pd.merge(
    ela_df, sela_df,
    on=['function_id', 'instance_id'],
    suffixes=('_ELA', '_SELA'),
    how='inner'
)

# ----------------------------------------
# 4. Features and statistics to analyze
# ----------------------------------------
stats = ["min", "median", "max", "sd", "domi"]
# Select only stable ELA features
features = [f for f in ela_df.columns if f in stable_sela_features]

# ----------------------------------------
# 5. Compute Spearman correlations
# Ignore NaN and ±inf automatically
# ----------------------------------------
results_spearman = []

for f in features:
    row = {'feature': f}
    for s in stats:
        sela_col = f"{f}_{s}"
        if sela_col in merged_df.columns:
            # Identify valid indices excluding NaN and ±inf
            valid_idx = merged_df[[f, sela_col]].replace([np.inf, -np.inf], np.nan).dropna().index
            if len(valid_idx) > 0:
                correlation = merged_df.loc[valid_idx, f].corr(
                    merged_df.loc[valid_idx, sela_col],
                    method='spearman'
                )
                row[s] = correlation
            else:
                row[s] = np.nan
        else:
            row[s] = np.nan
    results_spearman.append(row)

spearman_corr_df = pd.DataFrame(results_spearman).set_index('feature')

# ----------------------------------------
# 6. Plot heatmap
# ----------------------------------------
sns.set_context("talk", font_scale=1.0)
sns.set_style("white")

fig, ax = plt.subplots(figsize=(12, 19))
sns.heatmap(
    spearman_corr_df,
    annot=True,
    cmap='RdBu_r',
    center=0,
    fmt=".2f",
    linewidths=1,
    linecolor='white',
    square=False,
    ax=ax,
    annot_kws={"fontsize": 16, "fontweight": "bold"},
    cbar_kws={"aspect": 50}
)

# Labels and font adjustments
ax.set_xlabel('S-ELA Features', fontsize=20, fontweight='bold', labelpad=10)
ax.set_ylabel('ELA Features', fontsize=20, fontweight='bold', labelpad=10)
cbar = ax.collections[0].colorbar
cbar.ax.tick_params(labelsize=12)
plt.xticks(rotation=45, ha='right', fontsize=14)
plt.yticks(fontsize=14)
plt.tight_layout()

# Save figure in both PDF and PNG formats
plt.savefig('spearman_correlation_sela_ela.pdf', bbox_inches='tight', transparent=True)
plt.savefig('spearman_correlation_sela_ela.png', dpi=300, bbox_inches='tight', transparent=True)
plt.show()
